# Phase 2 · Resolve pathogen–host pairs and prepare PubMed queries

This phase expands the configured selectors into a saved, explicit pair list before acquisition. PubMed query construction is versioned and pair-aware. A preflight reports pair counts and query previews without making network calls.

Live PubMed search/fetch must be run through a configured acquisition adapter. The current V4 package provides the query builder and normalized run contract; the adapter boundary below is deliberately separate.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT, LOG, LOG_PATH = configure_notebook('02_runs_and_pubmed', 'pilot-v1')
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 11:25:12,526 INFO Notebook session started at 2026-11-02T11:25:12.526101+00:00


2026-11-02 11:25:12,526 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:25:12,526 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_runs_and_pubmed.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_runs_and_pubmed.log


## Step 1 · Load configuration and taxonomy references

In [2]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.runs import resolve_run, save_resolved_run
from pathogen_atlas_v4.pubmed import build_pair_query
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
LOG.info('Inputs loaded: hosts=%d pathogens=%d config=%s', len(hosts), len(pathogens), CONFIG['run']['run_id'])
display(CONFIG['run'])

2026-11-02 11:25:13,039 INFO Inputs loaded: hosts=5 pathogens=142201 config=pilot-v1


{'run_id': 'pilot-v1',
 'config_version': 'pilot-config-v1',
 'selection_seed': 'v4-pilot-v1',
 'pathogen_ids': ['bacteria:c714cb9064025c2e9256',
  'bacteria:29a1ade6bbea956000ea',
  'bacteria:938707dc76ddb1767252',
  'bacteria:86eac8eb3f8014f8697e',
  'bacteria:1b4227203d006fda2543',
  'bacteria:ecea9896f0043fa7ac87',
  'bacteria:d847281296f3c4ce85a7',
  'bacteria:26379d3501f6190eacd0',
  'bacteria:b3ac68fa9b139d3924c5',
  'bacteria:f29992ba7b3eeaf7a012',
  'fungi:b0fe2cc3a7217da9d6fe',
  'fungi:6d988fb1c0025389f3a4',
  'fungi:5de7e206033ddb930229',
  'fungi:cb2bfa7d30c888183eb1',
  'fungi:d0df36122c744d54d240',
  'fungi:220addd7959fe6e85398',
  'fungi:c8398d9b60288e0f22fc',
  'fungi:b3132702246a2c5eb735',
  'fungi:e5d3e5aff9408d1bc0bb',
  'fungi:1b7e5866a8b0bb42b9b3',
  'viruses:996a1edc1722e68f5f14',
  'viruses:647cf44f1f25525548ed',
  'viruses:974033d610b19598fb16',
  'viruses:b14f3747c19f824a1d89',
  'viruses:2739ba46288b00948cdc',
  'viruses:c86c57b20bc49ea9d59c',
  'viruses:ea77

## Step 2 · Resolve the exact pair set

In [3]:
host_groups = {key: group.host_id.astype(str).tolist() for key, group in hosts.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in pathogens.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(hosts.host_id.astype(str)),
                       known_pathogen_ids=set(pathogens.pathogen_id.astype(str)),
                       host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
RESOLVED_PATH = ROOT / 'data/v4/runs' / resolved.run_id / 'resolved_config.json'
save_resolved_run(CONFIG, resolved, RESOLVED_PATH)
print('Run:', resolved.run_id, '| generation rule:', resolved.pair_generation)
print('Hosts:', len(resolved.host_ids), '| pathogens:', len(resolved.pathogen_ids), '| pairs:', len(resolved.pairs))
print('Resolved configuration:', RESOLVED_PATH)
LOG.info('Resolved and saved run: hosts=%d pathogens=%d pairs=%d hash=%s', len(resolved.host_ids), len(resolved.pathogen_ids), len(resolved.pairs), resolved.resolved_config_hash)

2026-11-02 11:25:13,074 INFO Resolved and saved run: hosts=4 pathogens=30 pairs=120 hash=ad2a3ac820f09acbff65b41294f3a70ca9a4de181f4341d7e180173cd5abd5bd


Run: pilot-v1 | generation rule: explicit_pairs
Hosts: 4 | pathogens: 30 | pairs: 120
Resolved configuration: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/resolved_config.json


## Step 3 · Build and inspect pair-aware query text

In [4]:
host_rows = hosts.set_index('host_id')
pathogen_rows = pathogens.set_index('pathogen_id')
PREVIEW_ROWS = []
for pathogen_id, host_id in resolved.pairs[:10]:
    pathogen = pathogen_rows.loc[pathogen_id]
    host = host_rows.loc[host_id]
    host_aliases = host.get('aliases', [])
    if hasattr(host_aliases, 'tolist'):
        host_aliases = host_aliases.tolist()
    host_aliases = list(host_aliases) if host_aliases is not None else []
    query = build_pair_query([pathogen.accepted_name], [host.accepted_name, *host_aliases],
                             strategy_version=CONFIG['acquisition']['query_strategy_version'])
    PREVIEW_ROWS.append({'pathogen_id': pathogen_id, 'pathogen': pathogen.accepted_name,
                         'host_id': host_id, 'host': host.accepted_name, **query})
display(pd.DataFrame(PREVIEW_ROWS))
LOG.info('Generated %d pair query previews using strategy %s', len(PREVIEW_ROWS), CONFIG['acquisition']['query_strategy_version'])

,pathogen_id,pathogen,host_id,host,query,query_hash,query_version
0,bacteria:c714cb9064025c2e9256,Sulfurospirillum multivorans,ncbi:9913,Bos taurus,"(""Sulfurospirillum multivorans""[Title/Abstract...",e32907254bb1852a0129dbbada4cb254e09c8e3a67eef4...,pathogen-host-title-abstract-v1
1,bacteria:c714cb9064025c2e9256,Sulfurospirillum multivorans,ncbi:9940,Ovis aries,"(""Sulfurospirillum multivorans""[Title/Abstract...",2cea3aca0d44956d875c78390a27bc15b5f51ec1c25884...,pathogen-host-title-abstract-v1
2,bacteria:c714cb9064025c2e9256,Sulfurospirillum multivorans,ncbi:4577,Zea mays,"(""Sulfurospirillum multivorans""[Title/Abstract...",8e4d4267782c3e9dcb54519219f5d94247e5d894513021...,pathogen-host-title-abstract-v1
3,bacteria:c714cb9064025c2e9256,Sulfurospirillum multivorans,ncbi:4530,Oryza sativa,"(""Sulfurospirillum multivorans""[Title/Abstract...",a72962a7038be8398fd51f91130962cfd2a0cb3c31fdfa...,pathogen-host-title-abstract-v1
4,bacteria:29a1ade6bbea956000ea,Actinomyces oricola,ncbi:9913,Bos taurus,"(""Actinomyces oricola""[Title/Abstract]) AND (""...",9d9251fc8526af12302c689a5d71127a3e724d1ed70350...,pathogen-host-title-abstract-v1
5,bacteria:29a1ade6bbea956000ea,Actinomyces oricola,ncbi:9940,Ovis aries,"(""Actinomyces oricola""[Title/Abstract]) AND (""...",8eceb8c8c8d74d4db5605f5a0fa150f6268ea7ca61700e...,pathogen-host-title-abstract-v1
6,bacteria:29a1ade6bbea956000ea,Actinomyces oricola,ncbi:4577,Zea mays,"(""Actinomyces oricola""[Title/Abstract]) AND (""...",9bac0d0f1197885202b0013410fdb46216dc0b6e3f860c...,pathogen-host-title-abstract-v1
7,bacteria:29a1ade6bbea956000ea,Actinomyces oricola,ncbi:4530,Oryza sativa,"(""Actinomyces oricola""[Title/Abstract]) AND (""...",074905daab1a02c2f59a3d98faf2cf4893a25d18f580e2...,pathogen-host-title-abstract-v1
8,bacteria:938707dc76ddb1767252,Hephaestia caeni,ncbi:9913,Bos taurus,"(""Hephaestia caeni""[Title/Abstract]) AND (""Bos...",9dc1a93f4cbdba6360f47f610b7f9c65b7af5e4ec50d0d...,pathogen-host-title-abstract-v1
9,bacteria:938707dc76ddb1767252,Hephaestia caeni,ncbi:9940,Ovis aries,"(""Hephaestia caeni""[Title/Abstract]) AND (""dom...",c2df03958e14ccc8cbe190a026e82cec082ff925d15927...,pathogen-host-title-abstract-v1


2026-11-02 11:25:13,087 INFO Generated 10 pair query previews using strategy pathogen-host-title-abstract-v1


## Step 4 · Preflight counts and acquisition readiness

In [5]:
preflight = {'pairs_to_search': len(resolved.pairs), 'distinct_pathogens': len(resolved.pathogen_ids),
             'distinct_hosts': len(resolved.host_ids), 'query_strategy_version': CONFIG['acquisition']['query_strategy_version'],
             'network_calls_made': 0, 'adapter_available': False}
display(preflight)
LOG.info('PubMed preflight: %s', preflight)

{'pairs_to_search': 120,
 'distinct_pathogens': 30,
 'distinct_hosts': 4,
 'query_strategy_version': 'pathogen-host-title-abstract-v1',
 'network_calls_made': 0,
 'adapter_available': False}

2026-11-02 11:25:13,091 INFO PubMed preflight: {'pairs_to_search': 120, 'distinct_pathogens': 30, 'distinct_hosts': 4, 'query_strategy_version': 'pathogen-host-title-abstract-v1', 'network_calls_made': 0, 'adapter_available': False}


## Step 5 · Search and fetch selected literature (adapter step)

In [6]:
RUN_ACQUISITION = False
PUBMED_ADAPTER = None  # supply an adapter that checkpoints pair-scoped PMID results and retrieval state
if RUN_ACQUISITION:
    if PUBMED_ADAPTER is None:
        raise RuntimeError('Configure a PubMed acquisition adapter before enabling network calls.')
    raise NotImplementedError('Adapter integration owns request pacing, retries, checkpoints, and resume logging.')
else:
    print('Preflight complete; no PubMed requests were made.')
    LOG.info('Acquisition skipped: RUN_ACQUISITION=False')

2026-11-02 11:25:13,093 INFO Acquisition skipped: RUN_ACQUISITION=False


Preflight complete; no PubMed requests were made.


The acquisition adapter should log pair progress, retries, ETA, and checkpoint counts to the shared run log. It must save query text/hash, result PMIDs, fetch status, and sampling choices separately from disease-gate decisions.